# Generate a forecast with AIFS ENS v2

**Author**: ECMWF AIFS team

*This notebook was last tested and operational on 23/06/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was adapted for the DestinE [2026 Machine Learning for Earth System Modelling Course](https://learning.ecmwf.int/course/view.php?id=99) from the published [AIFS ENS v2 example on HuggingFace](https://huggingface.co/ecmwf/aifs-ens-2.0/blob/main/run_AIFS_ENS_v2.0.ipynb).

It builds directly on the [AIFS Single v2 notebook](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m4/run_AIFS_v2.0.ipynb) from the previous module: there you generated a single, deterministic global forecast with ECMWF [open data](https://www.ecmwf.int/en/forecasts/datasets/open-data) and [anemoi-inference](https://anemoi-inference.readthedocs.io/en/latest/); here we introduce **ensemble** forecasting with AIFS ENS v2.
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
You may try to run/access this notebook on the free online platforms linked below. Please note they are not officially supported by or linked with ECMWF.

**Important note:**
This notebook requires specific GPU access and several GB of writeable disk space. It has been tested on Colab with **L4 and A100** GPUs. These do not come with Google's free plan, and such GPUs are also not available for free on Binder. Therefore, if you wish to run this notebook yourself, the best thing to do is find your own suitable GPU-based system, and set it up and run it there. If you have access to (for example) GPUs on ECMWF's ATOS HPCF, you can run this notebook through ECMWF's [JupyterHub](https://jupyterhub.ecmwf.int/). Depending on your system, small additional adjustments may be needed to run the notebook, and we cannot help you with this.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/m5/run_AIFS_ENS_v2.0.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m5/run_AIFS_ENS_v2.0.ipynb)
[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/ecmwf-training/2026-ml-esm-training/main?binder_path=m5&labpath=m5/run_AIFS_ENS_v2.0.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/m5/run_AIFS_ENS_v2.0.ipynb)
<!-- 
::: -->


# Introduction

In the previous module you used **AIFS Single v2** to generate a single, deterministic global forecast. A single forecast, however, cannot tell you how confident to be in it. Ensemble forecasting addresses this by producing a set of plausible forecasts ("members") whose spread quantifies the uncertainty of the prediction.

AIFS ENS v2 is an *inherently stochastic* model: each time it runs it injects random noise inside the network, so the same initial conditions can produce different forecasts, a set of which together make up the ensemble.

In this session, we will:

* Load meteorological open-source data from ECMWF to use as initial conditions
* Generate a weather forecast using AIFS ENS v2 from ECMWF
* Build a small ensemble of forecasts and explore what its spread tells us about forecast uncertainty (optional)

## Prepare your environment

This notebook requires the following packages:
- Python (version 3.11 or 3.12)
- numpy
- matplotlib
- cartopy
- anemoi-inference[huggingface] (version=0.8.3)
- anemoi-models (version=0.11.2)
- anemoi-utils (version=0.4.35.post3)
- torch (version=2.7.0)
- torch-geometric (version=2.6.1)
- earthkit-regrid (version=0.5.1)
- ecmwf-opendata (version=0.3.29)
- earthkit-data (version <1.0.0)

This notebook also requires the following:
- Ampere GPUs or newer (this notebook has been tested in Colab using the **L4** and **A100** runtimes, and on ECMWF's ATOS HPCF **AC cluster**)
- Several GB of writable disk space

Known limitations:
- Even with compatible GPUs, this notebook may not work out-of-the-box on all systems. A compatible CUDA/PyTorch/FlashAttention environment is required. See https://github.com/Dao-AILab/flash-attention?tab=readme-ov-file#installation-and-features for more details.


# 1. Install dependencies


Run the lines below to install the required packages, or use the provided `requirements.txt` to install the dependencies in your environment.

### To run this notebook in Colab (L4 or A100 runtimes)

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m5/requirements.txt

# Install flash-attn from a pre-built wheel (no compilation required).
# Pre-built wheels are available for Python 3.11 and 3.12 with CUDA 12 and torch 2.7 on Linux x86_64.
# For other configurations we try to build from source, which requires the CUDA toolkit (nvcc).
import sys, subprocess

_base = "https://github.com/cathalobrien/get-flash-attn/releases/download/v0.1-alpha"
_wheels = {
    (3, 11): f"{_base}/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp311-cp311-linux_x86_64.whl",
    (3, 12): f"{_base}/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp312-cp312-linux_x86_64.whl",
}
_ver = sys.version_info[:2]
_wheel = _wheels.get(_ver)
if _wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    f"flash-attn @ {_wheel}"], check=True)
else:
    print(f"No pre-built wheel for Python {_ver[0]}.{_ver[1]} — building from source (requires nvcc)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn==2.7.4.post1",
                    "--no-build-isolation"], check=True)

### To run this notebook through JupyterHub on ECMWF's ATOS HPCF

If you have access to GPUs on ECMWF's ATOS HPCF, you can run this notebook through [JupyterHub](https://jupyterhub.ecmwf.int/). 

Select the **ECMWF ATOS HPCF - GPU partition**

**Note:** The quick-start method below installs packages in your personal Python package directory for the active Jupyter kernel. This is convenient for running the notebook, but it may affect other notebooks that use the same Python environment. We recommend creating a dedicated Python environment and registering it as a Jupyter kernel.

In [ ]:
# %pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/m5/requirements.txt
# %pip install -q --force-reinstall --no-deps "flash-attn @ https://github.com/cathalobrien/get-flash-attn/releases/download/v0.1-alpha/flash_attn-2.8.3+cu12torch2.7cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

# 2. Check runtime environment

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. This notebook requires a CUDA-capable GPU.\n"
        "  • On Colab: Runtime → Change runtime type → select L4 or A100 GPU.\n"
        "  • On ATOS HPCF: ensure you have requested GPU resources (AC cluster)."
    )

# See https://github.com/huggingface/transformers/issues/28188
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)

print("GPU:", gpu_name)
print(f"Compute capability: {major}.{minor}")

if major < 8:
    raise RuntimeError(
        f"FlashAttention requires Ampere (compute capability >= 8.0) or newer.\n"
        f"Your GPU ({gpu_name}) has compute capability {major}.{minor}.\n"
        "  • On Colab: switch to an L4 or A100 runtime."
    )

In [ ]:
try:
    import flash_attn
    print("FlashAttention:", flash_attn.__version__)
except ImportError:
    raise RuntimeError(
        "flash-attn is not installed. Re-run the install cell (Section 1) and restart the kernel.\n"
        "Ensure you are on a compatible GPU node before installing."
    )

# 3. Import packages

In [ ]:
import datetime
from collections import defaultdict

import numpy as np

import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient

Store downloaded ECMWF data in local cache:

In [ ]:
ekd.config.set({'cache-policy': 'user'})

# 4. Prepare retrieval of initial conditions

Initial conditions are the initial state used by the model.

### List of parameters to retrieve from ECMWF open data

In [ ]:
PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw", "sd"]
PARAM_SFC_FC = ["lsm", "z", "slor", "sdor"]
PARAM_SOIL =["vsw","sot"]
PARAM_WAVE =["wmb", "h1012", "h1214", "h1417", "h1721", "h2125", "h2530", "mwd", "cdww", "mwp", "swh"]
PARAM_PL = ["gh", "t", "u", "v", "w", "q"]
LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50, 10]
SOIL_LEVELS = [1,2]


### Choose open data source and set the latest available date

In [ ]:
SOURCE = "ecmwf" # Other options are: "azure", "aws", "ecmwf" or "google" 

DATE = OpendataClient(SOURCE).latest()
print("Initial date is", DATE)

### Create a function to download and interpolate data from the ECMWF Open Data API


In [ ]:
def get_open_data(param, levelist=[], number = None, **kwargs):
    fields = defaultdict(list)
    # Get the data for the current date and the previous date
    for date in [DATE - datetime.timedelta(hours=6), DATE]:
        if number is None:
            data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist, source = SOURCE, **kwargs)
        else:
            kwargs.setdefault("stream", "enfo")            
            data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist, number=[number], source = SOURCE, **kwargs)
        
        for f in data:
            # Open data is between -180 and 180, we need to shift it to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate the data to from 0.25 to N320
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Add the values to the list
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)

    # Create a single matrix for each parameter
    for param, values in fields.items():
        fields[param] = np.stack(values)

    return fields

### Store model input fields

We start from a single set of initial conditions. Setting `number = None` uses the ensemble **control** analysis; setting `number` to an integer between `1` and `50` would instead pull one of the IFS ensemble's *perturbed* initial conditions.

In operations, AIFS ENS is run 51 times — once from the control and 50 more times from these different perturbed initial conditions — to build a full ensemble. Here we download the control state only, and in Section 11 we will reuse it to build an illustrative ensemble.

In [ ]:
fields = {}
number = None  # None = ensemble control; 1–50 selects a perturbed member

# 5. Download initial conditions from ECMWF open data

Downloads can take several minutes and may be interrupted during peak times.

### Download surface fields

In [ ]:
fields.update(get_open_data(param=PARAM_SFC, number=number, levtype='sfc'))
fields.update(get_open_data(param=PARAM_SFC_FC, levtype='sfc')) # Add constant surface fields, retrieved from fc
assert all(p in fields for p in [*PARAM_SFC, *PARAM_SFC_FC]), "Missing parameters: %s" % (set(PARAM_SFC) + set(PARAM_SFC_FC) - set(fields.keys()))

### Download wave fields

In [ ]:
fields.update(get_open_data(param=PARAM_WAVE, stream="wave" if not number else "waef", number=number)) # Add wave fields, retrieved from waef if using ensemble data, or wave otherwise
assert all(p in fields for p in PARAM_WAVE), "Missing parameters: %s" % (set(PARAM_WAVE) - set(fields.keys()))

### Download soil fields

In [ ]:
soil=get_open_data(param=PARAM_SOIL,levelist=SOIL_LEVELS, number=number)

soil_names = [f"{p}_{lev}" for p in PARAM_SOIL for lev in SOIL_LEVELS]
assert all(p in soil for p in soil_names), "Missing parameters: %s" % (set(soil_names) - set(soil.keys()))

### Download pressure level fields

In [ ]:
fields.update(get_open_data(param=PARAM_PL, levelist=LEVELS, number=number))

PRESSURE_NAMES = [f"{p}_{lev}" for p in PARAM_PL for lev in LEVELS]
assert all(p in fields for p in PRESSURE_NAMES), "Missing parameters: %s" % (set(PRESSURE_NAMES) - set(fields.keys()))

# 6. Apply data transformations

AIFS ENS v2 expects the data to match the data it was trained on.

Transform the mean wave direction into sine and cosine components:

In [ ]:
mwd = fields.pop("mwd")
mwd_rad = np.deg2rad(mwd)

fields["cos_mwd"] = np.cos(mwd_rad)
fields["sin_mwd"] = np.sin(mwd_rad)

Rename soil fields:

In [ ]:
mapping = {'sot_1': 'stl1', 'sot_2': 'stl2',
           'vsw_1': 'swvl1','vsw_2': 'swvl2'}
for k,v in soil.items():
    fields[mapping[k]]=v

Remove unused specific humidity field:

In [ ]:
fields.pop("q_10", None);  # Remove the 10hPa level for specific humidity, as it is not used in the model

Apply land-sea mask:

In [ ]:
# Mask sea points (where the land-sea mask is 0). lsm was downloaded with the
# constant surface fields above, so we can use it directly.
mask = fields["lsm"][0].flatten() == 0

fields["sd"][:, mask] = np.nan
fields["swvl1"][:, mask] = np.nan
fields["swvl2"][:, mask] = np.nan

Convert geopotential height into geopotential

In [ ]:
# Transform GH to Z
for level in LEVELS:
    gh = fields.pop(f"gh_{level}")
    fields[f"z_{level}"] = gh * 9.80665

# 7. Create initial forecast state

In [ ]:
input_state = dict(date=DATE, fields=fields)

# 8. Load AIFS ENS v2 model


### Download the model checkpoint from Hugging Face

In [ ]:
checkpoint = {"huggingface":"ecmwf/aifs-ens-2.0"}

To reduce the memory usage of the model certain environment variables can be set, like the number of chunks of the model's mapper. Please refer to:
- https://anemoi.readthedocs.io/projects/models/en/latest/modules/layers.html#anemoi-inference-num-chunks
https://pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf

for more information. To do so, you can use the code below:
```
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True' 
os.environ['ANEMOI_INFERENCE_NUM_CHUNKS']='16'
```

### Create a runner

In [ ]:
runner = SimpleRunner(checkpoint, device='cuda')

**Note - changing the device from GPU to CPU**

- Running the transformer model used on the CPU is tricky, it depends on the FlashAttention library which only supports Nvidia and AMD GPUs, and is optimised for performance and memory usage
- In newer versions of anemoi-models, v0.4.2 and above, there is an option to switch off flash attention and uses Pytorchs Scaled Dot Product Attention (SDPA). The code snippet below shows how to overwrite a model from a checkpoint to use SDPA. Unfortunately it's not optimised for memory usage in the same way, leading to much greater memory usage. Please refer to https://github.com/ecmwf/anemoi-inference/issues/119 for more details 

# 9. Run the model to generate a forecast 

The example below is a 12-hour forecast for demonstration purposes. Change the LEAD_TIME to modify the forecast length.

### Run the forecast

In [ ]:
LEAD_TIME = 12

states = []

for state in runner.run(input_state=input_state, lead_time=LEAD_TIME):
    states.append(state)
    print_state(state)

### Note
Users should not expect this notebook to reproduce ECMWF operational AIFS forecasts exactly. This is due to two factors:

#### 1. GPU non-determinism
GPU operations are not always bitwise deterministic, meaning repeated runs can produce slightly different numerical results.

To enforce determinism at GPU level, the following settings can be configured:

```
#First, in a terminal
export CUBLAS_WORKSPACE_CONFIG=:4096:8

#And then before running inference:
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

```
Please note that using the above approach to enable deterministic behaviour will **significantly** increase runtime. 

#### 2. Differences in input data reprojection 
The initial conditions used in this notebook are downloaded from ECMWF open data. The data is provided on a 0.25° latitude/longitude grid and reprojected to the N320 grid for use by the AIFS.

In ECMWF's operational forecasting system, the initial conditions come directly from operational IFS analyses on the native o1280 grid and are reprojected directly to N320.

These differing reprojection pathways can introduce small differences in the model input fields, which may lead to minor differences in the resulting forecasts.

# 10. Optional: Inspect the forecast

### Plot a field (e.g. 100u)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as tri

In [ ]:
def fix(lons):
    # Shift the longitudes from 0-360 to -180-180
    return np.where(lons > 180, lons - 360, lons)

latitudes = states[-1]["latitudes"]
longitudes = states[-1]["longitudes"]
values = states[-1]["fields"]["100u"]


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")

triangulation = tri.Triangulation(fix(longitudes), latitudes)

contour=ax.tricontourf(triangulation, values, levels=20, transform=ccrs.PlateCarree(), cmap="RdBu")
cbar = fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label="100u")

plt.title("100m winds (100u) at {}".format(states[-1]["date"]))
plt.show()

# 11. Build an ensemble

AIFS ENS v2 is an inherently stochastic model: each time it runs, it injects random noise inside the neural network, so running the same initial conditions through it again yields a different forecast. A collection of such forecasts is an ensemble, and the spread between its members measures the forecast uncertainty. Let us exploit this stochasticity to build our own ensemble.

> ⚠️ **Note: This is a simplification for teaching, not how the operational ensemble is built**
>
> The operational AIFS ENS is built from 51 members: a control run, plus 50 forecasts started from *different, perturbed initial conditions* (the `number = 1…50` knob from Section 4). Each of these also receives the model's internal noise. Reproducing this flow here would mean downloading 51 separate sets of initial conditions. To keep this notebook fast and not overload the ECMWF servers with data requests, we hold the initial conditions fixed (the single control download above) and rely on the model's internal stochasticity alone to generate spread. Because it ignores initial-condition uncertainty this approach underestimates the true forecast spread.

Running several members takes a few minutes. `N_MEMBERS` and `LEAD_TIME` are set modestly below; increase them for a richer (but slower) ensemble.

In [ ]:
N_MEMBERS = 8        # operational AIFS ENS uses 51 members; we use a small number for speed
LEAD_TIME = 48       # forecast horizon in hours — a longer lead lets the members spread further apart
ENS_VARIABLE = "2t"  # the field we track across the ensemble (2 m temperature)

# We extract only ENS_VARIABLE (one of ~120 fields) to stay within memory; to plot a different variable,
# change ENS_VARIABLE and re-run this cell.
members = []      # one array per member, each of shape (steps, n_gridpoints)
valid_times = []  # forecast valid time of each step (identical across members)

for m in range(N_MEMBERS):
    steps, times = [], []
    for state in runner.run(input_state=input_state, lead_time=LEAD_TIME):
        steps.append(state["fields"][ENS_VARIABLE].copy())  # .copy() breaks the aliasing
        times.append(state["date"])
    members.append(np.stack(steps))
    valid_times = times
    latitudes, longitudes = state["latitudes"], state["longitudes"]
    print(f"Member {m + 1}/{N_MEMBERS} complete ({len(steps)} steps)")

ensemble = np.stack(members)  # shape: (members, steps, n_gridpoints)
lead_hours = [(t - input_state["date"]).total_seconds() / 3600 for t in valid_times]
print("Ensemble array:", ensemble.shape, "| lead times (h):", lead_hours)

# 12. Explore the forecast uncertainty

With several members we can ask not just *"what is the forecast?"* but *"how certain is it?"*. A common way to see this for a single location is an ensemble plume: the forecast trajectory of every member through time, overlaid. Where the members stay close together the forecast is confident; where they fan out it is uncertain.

Pick a location below — change `TARGET_LAT`/`TARGET_LON` to somewhere you care about (the default is De Bilt, in the Netherlands). To plot a different variable, set `ENS_VARIABLE` in the cell above and re-run it.

In [ ]:
TARGET_LAT, TARGET_LON = 52.10, 5.18  # De Bilt; change to a location you care about

# Find the nearest model grid point to the target location.
lon_pm = fix(longitudes)
idx = int(np.argmin((latitudes - TARGET_LAT) ** 2 + (lon_pm - TARGET_LON) ** 2))

# Time series at that grid point for every member: shape (members, steps).
series = ensemble[:, :, idx] - 273.15  # K -> °C (ENS_VARIABLE is 2 m temperature)
ens_mean = series.mean(axis=0)
ens_std = series.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
for member in series:
    ax.plot(lead_hours, member, color="grey", linewidth=0.8, alpha=0.6)
ax.plot(lead_hours, ens_mean, color="C3", linewidth=2.5, label="ensemble mean")
ax.fill_between(lead_hours, ens_mean - ens_std, ens_mean + ens_std,
                color="C3", alpha=0.2, label="±1 std (spread)")
ax.set_xlabel("Forecast lead time (hours)")
ax.set_ylabel("2 m temperature (°C)")
ax.set_title(f"Ensemble plume at ({TARGET_LAT}°N, {TARGET_LON}°E) — {N_MEMBERS} members")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Bonus: where is the forecast most uncertain?

The plume shows uncertainty at one location. We can also map it globally: at the final lead time, compute the **standard deviation across members** at every grid point. This reveals *where* the model is least sure of its 2 m temperature forecast — often along fast-moving weather systems and fronts, and generally growing with lead time.

In [ ]:
# Standard deviation across members at the final lead time — a map of where the
# 2 m temperature forecast is least certain.
spread = ensemble[:, -1, :].std(axis=0)  # (gridpoints,)

triangulation = tri.Triangulation(fix(longitudes), latitudes)

fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=":")
contour = ax.tricontourf(triangulation, spread, levels=20, transform=ccrs.PlateCarree(), cmap="viridis")
fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label="2 m temperature spread (K)")
plt.title(f"Ensemble spread (std of {N_MEMBERS} members) at +{int(lead_hours[-1])} h")
plt.show()